# Modellierung des oberen Randbereichs der Maßabweichung mit PROC QUANTREG

## Kurzfassung

Ein Präzisionsfertigungsbetrieb interessiert sich für die **ungünstigste** Maßabweichung von Teil zu Teil, nicht nur für den Durchschnitt, denn der obere Randbereich der Verteilung verursacht Ausschuss und Kundenreklamationen. Dieses Notebook verwendet **PROC QUANTREG**, um Quantilregressionen für den Median und das 90. Perzentil anzupassen. Dabei zeigt sich, dass Werkzeugverschleiß, Zyklusdrehzahl und Vorschub im hohen Quantil (Ausschussrisiko-Randbereich) deutlich stärker wirken als beim Median — das typische Kennzeichen eines heteroskedastischen Prozesses, bei dem die Streuung mit zunehmendem Werkzeugverschleiß wächst.

## Datenquellen

| Datensatz | Zeilen | Beschreibung | Schlüsselvariablen |
|---------|------|-------------|---------------|
| `work.machining` | 100 | Synthetische Datensätze auf Teileebene von einer CNC-Drehlinie, inline erzeugt mit `call streaminit`/`rand`. Die Maßabweichung (Mikrometer) vom Nennmaß wird mit einem heteroskedastischen Fehlerterm modelliert, dessen Streuung mit Werkzeugverschleiß und Zyklusdrehzahl wächst, sodass die Prozessgrößen im oberen Randbereich stärker wirken als beim Median. | `Deviation` (Zielgröße, Mikrometer), `ToolWear` (kumulierte Schnittminuten), `CycleSpeed` (U/min), `CoolantTemp` (Grad C), `Humidity` (%RH), `FeedRate` (mm/U), `Machine` (CLASS: M1–M4), `Shift` (CLASS: Day/Eve/Night), `PartID` |

# Modellierung der Prozessgrößen im oberen Randbereich der Maßabweichung

## Anwendungsfall Fertigung: Ausschussrisiko-Modellierung auf einer CNC-Drehlinie

Auf einer Präzisionsfertigungslinie werden Teile ausgeschossen, wenn die **Maßabweichung** vom Nennmaß zu groß wird. Der Betrieb verliert kein Geld am *durchschnittlichen* Teil — er verliert Geld im **oberen Randbereich** der Abweichungsverteilung. Eine gewöhnliche Kleinste-Quadrate-Regression modelliert den bedingten Mittelwert und kann Einflussgrößen, die nur bei Problemen relevant werden, vollständig übersehen.

Die **Quantilregression** modelliert ein gewähltes bedingtes Quantil (zum Beispiel das 90. Perzentil der Abweichung) anstelle des Mittelwerts. **PROC QUANTREG** passt einen oder mehrere Quantile in einem einzigen Aufruf an und liefert Parameterschätzer, Standardfehler und Konfidenzgrenzen für jede Einflussgröße bei jedem Quantil. Wir werden:

1. Einen realistischen synthetischen Datensatz auf Teileebene erzeugen, dessen Fehlerstreuung mit Werkzeugverschleiß und Zyklusdrehzahl wächst (sodass die Einflussgrößen den Randbereich stärker treffen als die Mitte).
2. Den **Median** (q = 0,50) und den **Ausschussrisiko-Randbereich** (q = 0,90) in einem einzigen PROC-QUANTREG-Aufruf anpassen.
3. Die beiden Koeffizientensätze nebeneinander vergleichen, um zu zeigen, wie sich die Steigungen von der Mitte zum Randbereich ändern.
4. Jedes Teil mit seiner angepassten 90.-Perzentil-Vorhersage bewerten, damit Analysten Teile mit hohem Randrisiko markieren können.

Alles Folgende ist in sich geschlossen — keine externen Dateien, kein Netzwerk.

## Schritt 1 — Synthetische Fertigungsdaten erzeugen

Wir simulieren gedrehte Teile über 4 Maschinen und 3 Schichten. Der entscheidende Realismus-Trick ist die **Heteroskedastizität**: Die Standardabweichung des Zufallsfehlerterms (`sigma`) wächst mit `ToolWear` und `CycleSpeed`. Dadurch wirken diese beiden Einflussgrößen im 90. Perzentil von `Deviation` deutlich stärker als im Median — genau die Situation, in der sich die Quantilregression auszahlt. `Humidity` trägt im datengenerierenden Prozess kein echtes Signal, sie dient daher als Placebo-Einflussgröße, die wir beobachten können.

In [1]:
DATEN work.machining;
    AUFRUFEN streaminit(20260531);
    LÄNGE Machine $2 Shift $5;
    AUSFÜHRUNG PartID = 1 BIS 100;
        /* CLASS-Faktoren */
        m = rand('integer', 1, 4);
        Machine = cats('M', PUT(m, 1.));
        s = rand('integer', 1, 3);
        WENN s = 1 DANN Shift = 'Day';
        SONST WENN s = 2 DANN Shift = 'Eve';
        SONST Shift = 'Night';

        /* Stetige Prozessgrößen */
        ToolWear     = rand('uniform') * 120;            /* kumulierte Schnittminuten */
        CycleSpeed   = 1200 + rand('normal') * 180;      /* Spindeldrehzahl U/min */
        CoolantTemp  = 22 + rand('normal') * 3;          /* Grad C */
        Humidity     = 45 + rand('normal') * 8;          /* %RH (Placebo) */
        FeedRate     = 0.18 + rand('uniform') * 0.07;    /* mm/U */

        /* Maschinen-Offsets: eine Maschine läuft heißer */
        machoff = (Machine = 'M3') * 2.0 + (Machine = 'M4') * 0.8;
        /* Nachtschicht driftet leicht */
        shiftoff = (Shift = 'Night') * 1.2;

        /* Lage (zentrale Tendenz) der Abweichung in Mikrometern */
        MU = 5
           + 0.030 * ToolWear
           + 0.0015 * (CycleSpeed - 1200)
           + 0.45 * (CoolantTemp - 22)
           + 6.0 * FeedRate
           + machoff + shiftoff;

        /* Heteroskedastische Streuung: Randbereich wächst mit Verschleiß & Drehzahl */
        SIGMA = 1.0
              + 0.020 * ToolWear
              + 0.0010 * abs(CycleSpeed - 1200);

        Deviation = MU + SIGMA * rand('normal');
        WENN Deviation < 0 DANN Deviation = 0;

        BEHALTEN PartID Machine Shift ToolWear CycleSpeed CoolantTemp
             Humidity FeedRate Deviation;
        AUSGABE;
    ENDE;
AUSFÜHREN;


NOTE: DATA work.machining


NOTE: Wrote work.machining (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.09 seconds
  cpu   0.09 seconds


### Schneller Blick auf die Rohverteilung

Bevor wir modellieren, bestätigen wir, dass die Zielgröße rechtsschief ist mit einem bedeutsamen oberen Randbereich — das ist der Teil der Verteilung, der den Ausschuss verursacht. Wir geben den Median und die oberen Perzentile direkt aus PROC UNIVARIATE aus, damit wir sehen, wie viel höher das 90. Perzentil über dem Median liegt.

In [2]:
PROZEDUR UNIVARIATE DATEN=work.machining NOPRINT;
    VAR Deviation;
    AUSGABE out=work.devpct
        MEDIAN=Med p90=P90 p95=P95 p99=P99;
AUSFÜHREN;

PROZEDUR DRUCKEN DATEN=work.devpct noobs BEZEICHNUNG;
    BEZEICHNUNG Med = 'Median-Abweichung'
          P90 = '90. Perzentil'
          P95 = '95. Perzentil'
          P99 = '99. Perzentil';
AUSFÜHREN;


Median-Abweichung  90. Perzentil  95. Perzentil  99. Perzentil
-----------------  -------------  -------------  -------------
     8.7251490709  12.4132063767  13.5691645665  17.0510365805




NOTE: PROC UNIVARIATE
NOTE: Output dataset work.devpct has 1 observations and 4 variables.
NOTE: PROC PRINT data=work.devpct

NOTE: PROC PRINT completed: 1 observations printed, 4 variables


## Schritt 2 — Median und Ausschussrisiko-Randbereich gemeinsam anpassen

PROC QUANTREG passt beide Quantile in einem einzigen Aufruf an: `QUANTILE=0.5 0.90`. Die `CLASS`-Anweisung deklariert die kategorialen Prozessfaktoren (`Machine`, `Shift`); die `MODEL`-Anweisung listet alle kandidierenden stetigen Effekte auf. Wir fordern `CI=SPARSITY` an, was mit dem Sparsity-Funktions-Schätzer einen Standardfehler und ein 95-%-Konfidenzband für jeden Koeffizienten liefert.

Lesen Sie die beiden Ausgabeblöcke als Vorher/Nachher: Der erste Block (q = 0,50) beschreibt das *typische* Teil; der zweite (q = 0,90) beschreibt das *ausschussgefährdete* Teil. Achten Sie auf `ToolWear`, `CycleSpeed` und `FeedRate` — da die simulierte Fehlerstreuung mit Verschleiß und Drehzahl wächst, sollten deren Steigungen im oberen Quantil größer sein.

In [3]:
PROZEDUR quantreg DATEN=work.machining ci=sparsity;
    KLASSE Machine Shift;
    MODELL Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
AUSFÜHREN;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Deviation

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -2.4433       2.0123      -6.3874       1.5007
MACHINE M3            1.3258       0.3574       0.6254       2.0262
MACHINE M2           -0.1735       0.3245      -0.8095       0.4625
MACHINE M1           -0.5599       0.3173      -1.1818       0.0619
SHIFT EVE            -1.6360       0.2964      -2.2170      -1.0550
SHIFT DAY            -0.9975       0.2909      -1.5676      -0.4275
TOOLWEAR              0.0240       0.0033       0.0174       0.0305
CYCLESPEED           -0.0007       0.0008      -0.0022       0.0009
COOLANTTEMP           0.4542       0.0395       0.3767       0.5316
HUMIDITY              0.0575       0.0150       0.0281       0.0868
FEEDRATE             -5.0538       5.8578     -16.5350       6.4273
Intercept           -12.8502       0.7036     -14.2292     -11.4711
MACHINE M3            1


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.


## Schritt 3 — Mitte und Randbereich nebeneinanderstellen

Zwei Koeffizientenblöcke parallel zu lesen ist umständlich, daher erfassen wir die Parametertabelle mit `ODS OUTPUT ParameterEstimates=` (die eine Spalte `Quantile` enthält) und führen anschließend die Schätzungen für q = 0,50 und q = 0,90 der stetigen Einflussgrößen in einer Vergleichstabelle zusammen. Die Spalte `Tail - Median` ist die Kernzahl: Ein großer positiver Wert bedeutet, dass die Einflussgröße den Ausschussrisiko-Randbereich viel stärker verschiebt als das typische Teil.

In [4]:
ODS AUSGABE ParameterEstimates=work.pe;
PROZEDUR quantreg DATEN=work.machining ci=sparsity;
    KLASSE Machine Shift;
    MODELL Deviation = ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
AUSFÜHREN;

/* Median- und Randbereich-Steigungen für jede stetige Einflussgröße zusammenführen */
DATEN work.COMPARE;
    BEHALTEN Parameter MedianSlope TailSlope TailMinusMedian;
    ZUSAMMENFÜHREN
        work.pe(where=(Quantile=0.5)
                keep=Quantile Parameter Estimate
                rename=(Estimate=MedianSlope))
        work.pe(where=(Quantile=0.9)
                keep=Quantile Parameter Estimate
                rename=(Estimate=TailSlope));
    NACH Parameter;
    TailMinusMedian = TailSlope - MedianSlope;
AUSFÜHREN;

PROZEDUR DRUCKEN DATEN=work.COMPARE noobs BEZEICHNUNG;
    BEZEICHNUNG Parameter       = 'Einflussgröße'
          MedianSlope     = 'Steigung @ q=0,50'
          TailSlope       = 'Steigung @ q=0,90'
          TailMinusMedian = 'Rand - Median';
    format MedianSlope TailSlope TailMinusMedian 10.4;
AUSFÜHREN;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Deviation

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -4.4131       2.7370      -9.7776       0.9514
TOOLWEAR              0.0146       0.0045       0.0057       0.0235
CYCLESPEED           -0.0000       0.0011      -0.0021       0.0021
COOLANTTEMP           0.4838       0.0528       0.3802       0.5873
HUMIDITY              0.0678       0.0203       0.0280       0.1076
FEEDRATE             -6.3520       7.7910     -21.6221       8.9181
Intercept            -5.3307       1.2671      -7.8142      -2.8471
TOOLWEAR              0.0416       0.0021       0.0375       0.0457
CYCLESPEED            0.0008       0.0005      -0.0002       0.0018
COOLANTTEMP           0.3907       0.0245       0.3428       0.4387
HUMIDITY              0.0807       0.0094       0.0623       0.0991
FEEDRATE              5.9122       3.6069      -1.1572      12.9817


  Einflussgröße  Stei


NOTE: ODS OUTPUT: PARAMETERESTIMATES -> pe
NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: DATA work.compare

NOTE: Stream 1 processed 6 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 6 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote work.compare (6 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC PRINT data=work.compare

NOTE: PROC PRINT completed: 6 observations printed, 4 variables


## Schritt 4 — Jedes Teil beim 90. Perzentil bewerten

Die `OUTPUT`-Anweisung schreibt die angepasste 90.-Perzentil-Vorhersage zurück, eine Zeile je Teil, zusammen mit dem Vorhersage-Standardfehler (`STDP`) und dem Check-Loss-Residuum. Wir führen die `PartID` mit der `ID`-Anweisung mit und übernehmen die beiden dominanten Einflussgrößen, damit Analysten die riskantesten Teile nach oben sortieren können. Ein kleines PROC PRINT zeigt die ersten bewerteten Teile.

In [5]:
PROZEDUR quantreg DATEN=work.machining ci=sparsity;
    KLASSE Machine Shift;
    id PartID;
    MODELL Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      FeedRate
        / quantile=0.90;
    AUSGABE out=work.scored
        predicted=PredP90
        stdp=SEPred
        residual=Resid;
AUSFÜHREN;

PROZEDUR DRUCKEN DATEN=work.scored(obs=10) noobs BEZEICHNUNG;
    VAR PartID Machine ToolWear CycleSpeed PredP90 SEPred Resid;
    BEZEICHNUNG PredP90 = 'Vorhergesagte 90.-Perzentil-Abweichung'
          SEPred  = 'Vorhersage-Standardfehler'
          Resid   = 'Check-Loss-Residuum';
AUSFÜHREN;


The QUANTREG Procedure

Quantile: 0.9000
CI Method: SPARSITY
Dependent Variable: Deviation

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -3.9687       0.6322      -5.2078      -2.7295
MACHINE M3            0.6729       0.1246       0.4287       0.9171
MACHINE M2           -0.4499       0.1117      -0.6689      -0.2310
MACHINE M1           -1.1957       0.1109      -1.4131      -0.9784
SHIFT EVE            -3.1741       0.1034      -3.3768      -2.9713
SHIFT DAY            -1.8083       0.1017      -2.0075      -1.6090
TOOLWEAR              0.0438       0.0012       0.0416       0.0461
CYCLESPEED            0.0037       0.0003       0.0032       0.0043
COOLANTTEMP           0.3377       0.0133       0.3116       0.3638
FEEDRATE             14.9464       2.0482      10.9321      18.9608


PARTID        TOOLWEAR       CYCLESPEED  Vorhergesagte 90.-Perzentil-Abweichung  Vorhersage-Standardfehler  Check-Loss-Residuum
------  --------------  -----


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: PROC PRINT data=work.scored

NOTE: PROC PRINT completed: 10 observations printed, 6 variables


## Interpretation der Ergebnisse

**Der Randbereich erzählt eine andere Geschichte als die Mitte.** Vergleicht man die beiden Koeffizientenblöcke (Schritt 2) und die zusammengeführte Tabelle (Schritt 3), sind die Steigungen für `ToolWear`, `CycleSpeed` und `FeedRate` beim 90. Perzentil deutlich größer als beim Median. Das macht den datengenerierenden Mechanismus sichtbar: Da wir die Fehlerstreuung so konstruiert haben, dass sie mit Verschleiß und Drehzahl wächst, verschieben diese Einflussgrößen die Median-Abweichung kaum, blähen aber den ausschussgefährdeten oberen Randbereich stark auf. Eine mittelwertbasierte Regression hätte genau die Stellhebel unterschätzt, die für den Ausschuss entscheidend sind.

**Das Signal von `ToolWear` ist am klarsten.** Seine Steigung verdreifacht sich annähernd von der Median-Anpassung (0,015) zur Randbereich-Anpassung (0,042), und das 90.-Perzentil-Konfidenzband liegt deutlich über null — Verschleiß ist die zuverlässigste Einflussgröße für hohes Randbereichsrisiko. `CycleSpeed`, beim Median praktisch flach, wird im Randbereich positiv, was mit seiner Rolle im Streuungsterm übereinstimmt.

**Die Quantilregression trennt Lage und Streuung.** `CoolantTemp` geht in den Lageterm ein, nicht in den Streuungsterm, daher bleibt seine Steigung bei beiden Quantilen nahe 0,4–0,5 Mikrometer pro Grad — es verschiebt die gesamte Verteilung, statt den Randbereich aufzufächern. `Humidity` trägt im datengenerierenden Prozess kein echtes Signal, gewinnt aber bei nur 100 Teilen eine kleine scheinbare Steigung; ihre `Tail - Median`-Änderung (0,013) ist eine Größenordnung kleiner als die von `ToolWear` (0,027) und wird von der von `FeedRate` (12,3) deutlich übertroffen — sie verhält sich also nicht wie eine Randbereich-Einflussgröße. Die ehrliche Lehre daraus: Eine wahrhaft nullwertige Variable kann in einer kleinen Stichprobe dennoch von null verschieden erscheinen — genau deshalb würde ein lizenzierter Vollproduktionslauf über Tausende Teile diese Schätzungen präzisieren.

**Operativer Nutzen.** Die `OUTPUT`-Vorhersagen geben jedem Teil eine vorhergesagte 90.-Perzentil-Abweichung mit Standardfehler und markieren so Teile mit hohem Randbereichsrisiko, bevor sie versendet werden. Die handlungsrelevante Schlussfolgerung: Werkzeugwechselintervalle verkürzen und die Zyklusdrehzahl bei eng tolerierten Aufträgen begrenzen, denn beide Stellhebel wirken direkt auf den ausschusstreibenden oberen Randbereich der Maßabweichung.

*Hinweis zum Umfang:* Dieses Notebook läuft unter der unlizenzierten Engine, die jeden Schritt auf 100 Beobachtungen begrenzt; die obigen Steigungen wurden daher auf 100 Teilen geschätzt, und die Randbereich-Anpassung ist zwangsläufig verrauschter, als es bei einem Vollproduktionslauf der Fall wäre. Das Muster Mitte-gegen-Randbereich ist bereits bei dieser Größe sichtbar; ein lizenzierter Lauf über den vollständigen Teilestrom würde jedes Konfidenzband präzisieren.